# Binarization with scBoolSeq

1) Binarize the matrix, the workflow is based on the raw matrix, and the cell type and the tiempoint are binarized separately
2) Evaluate the binarization result

In [1]:
import sys
#!pip install scboolseq
import scanpy as sc
import numpy as np
import pandas as pd
from scboolseq import scBoolSeq
import gc

In [2]:
adata = sc.read("/home/a.blanc-boekholt/Documents/Singlecell-R/Scripts/Matrix/cll_myel_raw_p2.h5ad")

/home/a.blanc-boekholt/miniconda3/lib/python3.13/site-packages/anndata/_io/h5ad.py:266: FutureWarning: Moving element from .uns['neighbors']['distances'] to .obsp['distances'].

This is where adjacency matrices should go now.
  return AnnData(**{


In [3]:
adata.X.max()

np.float64(7288.0)

In [4]:
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

In [5]:
adata.X.max()

np.float64(8.008315932903622)

1) Binarization with scBoolSeq

In [16]:
final_annot = ['Bridge', 'B naive', 'B intermediate', 'B memory', "B"]
timepoints = ['D1', 'D4', 'D8','D11', 'D14']
adata.obs['timepoint'] = adata.obs['timepoint'].astype('category')

# Make sure that logcounts is in X
#adata.X = adata.layers['logcounts']

# Final results : {cell_type: {timepoint: Series(gene -> 0/1/NaN)}}
binarized_states = {}

for ct in final_annot:
    print(f"\n{'='*50}")
    print(f"Processing {ct}...")
    adata_ct = adata[adata.obs['Annotation'] == ct].copy()
    n_cells = adata_ct.n_obs
    print(f" {n_cells} cells")
    adata_ct.obs['timepoint'] = adata_ct.obs['timepoint'].cat.remove_unused_categories()

    present_timepoints = adata_ct.obs['timepoint'].cat.categories.tolist()
    print(f" Timepoints présents : {present_timepoints}")
    # STEP 1 : HVG adapted to the cell type size
    n_top = min(2000, adata_ct.n_vars - 1) # jamais plus de gènes qu'on a
    # Check whether batch_key is valid (enough cells per batch)
    min_cells_per_tp = adata_ct.obs['timepoint'].value_counts().min()
    n_batches = len(present_timepoints)
    if min_cells_per_tp < 10 or n_batches < 2:
        print(f" ⚠ Too few cells per timepoint ({min_cells_per_tp})",f"or too few batches ({n_batches}) fitting without batch_key")
        sc.pp.highly_variable_genes(adata_ct, n_top_genes=n_top)
    else:
        sc.pp.highly_variable_genes(adata_ct, n_top_genes=n_top,
                                     batch_key='timepoint')
    adata_ct_hvg = adata_ct[:, adata_ct.var['highly_variable']].copy()
    print(f" {adata_ct_hvg.n_vars} HVGs selected")

    # STEP 2 : Building the DataFrame for scBoolSeq
    X_full = adata_ct_hvg.X
    if not isinstance(X_full, np.ndarray):
        X_full = X_full.toarray()
    expr_df_full = pd.DataFrame(
        X_full,
        index=adata_ct_hvg.obs_names,
        columns=adata_ct_hvg.var_names
    )
    # Eliminate the 0 genes
    expr_df_full = expr_df_full.loc[:, (expr_df_full != 0).any(axis=0)]
    print(f" {expr_df_full.shape[1]} genes after removing all-zero")

    # STEP 3 : Fit scBoolSeq on all the cell type
    #scbool = scBoolSeq()
    scbool = scBoolSeq(
        zeroinf_binarizer="quantile", # use quantile thresholds for zero-inflated genes
        margin_quantile=0.1, # 10th/90th percentile instead of 5th/95th
        dor_threshold=0.85, # classify as zero-inflated if >85% zeros
        alpha=0.0, # no IQR expansion (keep it simple)
    )
    scbool.fit(expr_df_full)
    print(f" scBoolSeq fitted")

    # STEP 4 : Binarize per time point
    binarized_states[ct] = {}
    for tp in timepoints:
        mask = adata_ct_hvg.obs['timepoint'] == tp
        if mask.sum() == 0:
            print(f" {tp}: no cells, skipping")
            continue
        subset = adata_ct_hvg[mask]
        X_tp = subset.X
        if not isinstance(X_tp, np.ndarray):
            X_tp = X_tp.toarray()
        expr_df_tp = pd.DataFrame(
            X_tp,
            index=subset.obs_names,
            columns=subset.var_names
        )
        # Align the genes with those of the fit
        expr_df_tp = expr_df_tp[expr_df_full.columns]
        # Binarize
        binarized = scbool.binarize(expr_df_tp)

        # STEP 5 : Appoint by majority vote
        # For each gene: 1 if the majority of cells = 1, 0 if the majority = 0
        # NaN if there is too much uncertainty (can choose your threshold)
        def majority_vote(col):
            valid = col.dropna()
            if len(valid) == 0:
                return np.nan
            frac_ones = (valid == 1).mean()
            frac_zeros = (valid == 0).mean()
            if frac_ones >= 0.5:
                return 1
            elif frac_zeros >= 0.5:
                return 0
            else:
                return np.nan

        aggregated = binarized.apply(majority_vote, axis=0)
        binarized_states[ct][tp] = aggregated
        n_ones = (aggregated == 1).sum()
        n_zeros = (aggregated == 0).sum()
        n_nan = aggregated.isna().sum()
        print(f" {tp}: {mask.sum()} cells → {n_ones} genes=1, {n_zeros} genes=0, {n_nan} genes=NaN")

    del adata_ct, adata_ct_hvg, expr_df_full, scbool
    gc.collect()

print("\nDone!")


Processing Bridge...
 264 cells
 Timepoints présents : ['D1', 'D11', 'D14', 'D4', 'D8']
 ⚠ Too few cells per timepoint (1) or too few batches (5) fitting without batch_key
 2001 HVGs selected
 2001 genes after removing all-zero


/home/a.blanc-boekholt/miniconda3/lib/python3.13/site-packages/diptest/diptest.py:170: UserWarning: Dip test is not valid for n <= 3
  warnings.warn("Dip test is not valid for n <= 3", stacklevel=1)
/home/a.blanc-boekholt/miniconda3/lib/python3.13/site-packages/numpy/_core/_methods.py:222: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/a.blanc-boekholt/miniconda3/lib/python3.13/site-packages/numpy/_core/_methods.py:214: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/a.blanc-boekholt/miniconda3/lib/python3.13/site-packages/statsmodels/nonparametric/kde.py:593: RuntimeWarning: invalid value encountered in cast
  binned = fast_linbin(x, a, b, gridsize) / (delta * nobs)


 scBoolSeq fitted
 D1: 1 cells → 88 genes=1, 86 genes=0, 1827 genes=NaN
 D4: 11 cells → 269 genes=1, 83 genes=0, 1649 genes=NaN
 D8: 55 cells → 522 genes=1, 61 genes=0, 1418 genes=NaN
 D11: 102 cells → 518 genes=1, 68 genes=0, 1415 genes=NaN
 D14: 95 cells → 524 genes=1, 62 genes=0, 1415 genes=NaN

Processing B naive...
 10768 cells
 Timepoints présents : ['D1', 'D11', 'D14', 'D4', 'D8']
 2000 HVGs selected
 2000 genes after removing all-zero


/home/a.blanc-boekholt/miniconda3/lib/python3.13/site-packages/diptest/diptest.py:170: UserWarning: Dip test is not valid for n <= 3
  warnings.warn("Dip test is not valid for n <= 3", stacklevel=1)


 scBoolSeq fitted
 D1: 1177 cells → 485 genes=1, 220 genes=0, 1295 genes=NaN
 D4: 1358 cells → 523 genes=1, 183 genes=0, 1294 genes=NaN
 D8: 2197 cells → 495 genes=1, 211 genes=0, 1294 genes=NaN
 D11: 2574 cells → 484 genes=1, 222 genes=0, 1294 genes=NaN
 D14: 3462 cells → 450 genes=1, 256 genes=0, 1294 genes=NaN

Processing B intermediate...
 28607 cells
 Timepoints présents : ['D1', 'D11', 'D14', 'D4', 'D8']
 2000 HVGs selected
 2000 genes after removing all-zero


...

 scBoolSeq fitted
 D1: 6242 cells → 672 genes=1, 165 genes=0, 1163 genes=NaN
 D4: 5138 cells → 712 genes=1, 126 genes=0, 1162 genes=NaN
 D8: 5295 cells → 672 genes=1, 166 genes=0, 1162 genes=NaN
 D11: 4893 cells → 661 genes=1, 177 genes=0, 1162 genes=NaN
 D14: 7039 cells → 636 genes=1, 202 genes=0, 1162 genes=NaN

Processing B memory...
 46 cells
 Timepoints présents : ['D1', 'D11', 'D14', 'D4', 'D8']
 ⚠ Too few cells per timepoint (2) or too few batches (5) fitting without batch_key
 2000 HVGs selected
 2000 genes after removing all-zero


/home/a.blanc-boekholt/miniconda3/lib/python3.13/site-packages/diptest/diptest.py:170: UserWarning: Dip test is not valid for n <= 3
  warnings.warn("Dip test is not valid for n <= 3", stacklevel=1)
/home/a.blanc-boekholt/miniconda3/lib/python3.13/site-packages/numpy/_core/_methods.py:222: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/a.blanc-boekholt/miniconda3/lib/python3.13/site-packages/numpy/_core/_methods.py:214: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/a.blanc-boekholt/miniconda3/lib/python3.13/site-packages/statsmodels/nonparametric/kde.py:593: RuntimeWarning: invalid value encountered in cast
  binned = fast_linbin(x, a, b, gridsize) / (delta * nobs)


 scBoolSeq fitted
 D1: 13 cells → 679 genes=1, 33 genes=0, 1288 genes=NaN
 D4: 2 cells → 95 genes=1, 19 genes=0, 1886 genes=NaN
 D8: 7 cells → 586 genes=1, 25 genes=0, 1389 genes=NaN
 D11: 8 cells → 614 genes=1, 22 genes=0, 1364 genes=NaN
 D14: 16 cells → 914 genes=1, 44 genes=0, 1042 genes=NaN

Processing B...
 215 cells
 Timepoints présents : ['D1', 'D11', 'D14', 'D4', 'D8']
 2000 HVGs selected
 2000 genes after removing all-zero


/home/a.blanc-boekholt/miniconda3/lib/python3.13/site-packages/diptest/diptest.py:170: UserWarning: Dip test is not valid for n <= 3
  warnings.warn("Dip test is not valid for n <= 3", stacklevel=1)


 scBoolSeq fitted
 D1: 27 cells → 1183 genes=1, 15 genes=0, 802 genes=NaN
 D4: 21 cells → 1203 genes=1, 12 genes=0, 785 genes=NaN
 D8: 41 cells → 1304 genes=1, 5 genes=0, 691 genes=NaN
 D11: 44 cells → 1312 genes=1, 5 genes=0, 683 genes=NaN
 D14: 82 cells → 1318 genes=1, 4 genes=0, 678 genes=NaN

Construction de la matrice finale...
Matrice finale : 25 observations × 6184 gènes
gene                A1BG  A2M  AACS  AAGAB  AAK1  AAMP  AAR2  AARS2  ABAT  \
observation                                                                 
Bridge_D1            NaN  NaN   NaN    NaN   NaN   NaN   NaN    NaN   NaN   
Bridge_D4            NaN  NaN   NaN    NaN   NaN   NaN   NaN    NaN   NaN   
Bridge_D8            1.0  NaN   NaN    NaN   NaN   NaN   NaN    NaN   NaN   
Bridge_D11           1.0  NaN   NaN    NaN   NaN   NaN   NaN    NaN   NaN   
Bridge_D14           1.0  NaN   NaN    NaN   NaN   NaN   NaN    NaN   NaN   
B naive_D1           NaN  NaN   NaN    NaN   NaN   NaN   NaN    NaN   NaN   
B n

In [17]:
aggregated.head()

TNFRSF4    1.0
ATAD3B     NaN
CEP104     1.0
PHF13      NaN
DNAJC11    1.0
dtype: float64